In [61]:
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go



from energies.bend import Bend
from energies.bend_twist import BendTwist
from energies.gravity import Gravity
from energies.random import RandomForce
from energies.twist import Twist
from math_util.rotation import RotationUtil, Quaternion
from math_util.vectors import Vector
from rod.RodHelixConverter import RodHelixConverter
from rod.helix import Helix
from rod.helix_util import HelixUtil
from rod.preprocess import Preprocess
from rod.rod_generator import RodGenerator
from rod.rod_util import RodUtil
from solver.sim import Sim
from visualization.visualizer import Visualizer

In [62]:
def strands_to_one_objs(strands: np.ndarray, frame_idx: int, output_file: str = None, y_up: bool = True):
    output_file = f"output/figure_samples/obj_{frame_idx}.obj" if output_file is None else output_file
    Visualizer.clear_output_file(output_file)
    vertex_offset = 1
    for strand in strands:
        pos = strand[:, :3]
        vertex_offset = Visualizer.to_simple_obj(pos=pos, output_file=output_file, init_offset=vertex_offset, y_up=y_up)
    return

def add_twist(pos, theta, twists_per_curl, angular_curl_freq):
    """
    Add twist to a rod based on desired twists per curl.

    Parameters:
    - pos: vertex positions (n+2, 3)
    - theta: twist angles (n+1,)
    - twists_per_curl: desired number of twists per curl
    - angular_curl_freq: angular frequency of the helix (in radians per index)
    """
    helix = RodHelixConverter.rod_to_helix(pos=pos, theta=theta)
    num_vertices = len(pos)
    num_twist_indices = len(helix.q) // 3

    # calculate number of indices per curl
    indices_per_curl = int(2 * np.pi / angular_curl_freq)
    print(f"indices per curl: {indices_per_curl}")

    # compute total twist indices to apply
    twist_rate = twists_per_curl / indices_per_curl
    total_twists = int(twist_rate * num_vertices)

    # select random twist indices and add twist
    twist_indices = np.random.choice(np.arange(num_twist_indices), size=total_twists, replace=False)
    q_indices = 3 * twist_indices
    helix.q[q_indices] += np.random.uniform(-30, 30, size=total_twists)

    return RodHelixConverter.helix_to_rod(helix=helix)

def add_twist_gaussian(pos, theta, twists_per_curl, angular_curl_freq):
    helix = RodHelixConverter.rod_to_helix(pos=pos, theta=theta)
    num_vertices = len(pos)
    num_twist_indices = len(helix.q) // 3

    # calculate number of indices per curl
    indices_per_curl = int(2 * np.pi / angular_curl_freq)

    # compute total twist indices to apply
    twist_rate = twists_per_curl / indices_per_curl
    total_twists = int(twist_rate * num_vertices)

    # select random twist indices and add twist
    twist_indices = np.random.choice(np.arange(num_twist_indices), size=total_twists, replace=False)
    twist_amplitudes = np.random.uniform(-30,  30, size=total_twists)
    twist_std = 0.1 * num_vertices
    
    for i in range(num_twist_indices):
        total_twist = 0.0
        for center, amp in zip(twist_indices, twist_amplitudes):
            gaussian = amp * np.exp(-0.5 * ((i - center) / twist_std) ** 2)
            total_twist += gaussian
        # scale amplitude to match desired total twists
        helix.q[3 * i] += total_twist * (2 * np.pi * twists_per_curl / len(twist_indices))

    return RodHelixConverter.helix_to_rod(helix=helix)

def in_to_meters(x):
    inches_to_meters = 0.0254
    return x * inches_to_meters



In [ ]:
# making figures for paper
from rod.rod_generator import RodGenerator

# base parameters: curl radius: .5", curl wavelength: 1", 
# twist prevalence: low but still existent like 1 or 2 per curl? 
# hair thickness: whatever you've been using

# 5 different deviations from that, for example: curl radius at 0 .25, .5, .75, 1

# def example_rod(n: int, curl_radius=1.5, curl_frequency=0.3, height_scale=0.5):
radii = [0.25, 0.5, 0.75, 1.0, 1.25] # inches
curl_freqs = []
curl_freqs = [0.2, 0.4, 0.6, 0.8, 1.0] # check units
twist_freqs = [0.0, 0.25, 0.5, 0.75] # 1 / inches roughly
masses = [0.8, 0.9, 1.0, 1.1, 1.2] # scaled

mass_default = 1.0
rad_default = 0.5
ang_curl_freq_default = 0.1 * np.pi
twist_prev_default = 10 # twists per curl

# sim parameters
beta = 0.1 # bending stiffness
k = 0.0
g = 9.81e-3
damping = 0.2
dt = 0.08 #0.04
xpbd_steps = 10
energies = [Gravity(), Bend(), Twist(), BendTwist()]

poses, thetas = [], []
sims = []
strand_labels = []

# Varying MASS
for i in range(1):
    
    strand_labels.append({'r': rad_default, 'f': ang_curl_freq_default, 'tf': twist_prev_default, 'm': masses[i]})
    print(strand_labels[i])
    pos, theta = RodGenerator.example_rod(n=300, curl_radius=in_to_meters(rad_default), curl_frequency=ang_curl_freq_default, height_scale=in_to_meters(0.05))
    pos, theta = add_twist(pos, theta, twist_prev_default, ang_curl_freq_default)
    pos[:, 1] -= pos[0, 1] # normalizing y
    pos[:, 2] -= pos[0, 2] # normalizing z
    pos[:, 0] += 10 * i

    n_sites, n_edges = pos.shape[0], theta.shape[0]
    mass = np.full((n_sites), masses[i]) * 1

    B = np.zeros((n_edges, 2, 2))
    B[:, 0, 0] = 1.0
    B[:, 1, 1] = 1.0
    
    frozen_pos_indices = np.array([0], dtype=int)
    frozen_theta_indices = np.array([], dtype=int)

    sim = Sim(pos=pos, theta=theta, B=B, beta=beta, k=k, g=g, mass=mass, energies=energies,
                damping=damping, dt=dt, xpbd_steps=xpbd_steps, frozen_pos_indices=frozen_pos_indices,
                frozen_theta_indices=frozen_theta_indices)
    sim.define_rest_state(pos=pos, theta=theta)
    
    poses.append(pos)
    thetas.append(theta)
    sims.append(sim)

    strands_to_one_objs(np.array(poses), i+1)
# poses, _ = add_twist(poses, theta, 0.7)

# poses = np.expand_dims(poses, axis=0)

# strands_to_one_objs(poses, frame_idx=6)

{'r': 0.5, 'f': 0.3141592653589793, 'tf': 10, 'm': 0.8}
indices per curl: 20


In [83]:
def plot_cls(poses):
    """
    Creates an interactive visualization of multiple centerlines.

    Parameters:
    - poses: [(n_sites, 3)] Centerline positions
    """
    # Create figure
    fig = go.Figure()

    # Plot each helix
    for idx, pos in enumerate(poses):
        # Plot centerline
        fig.add_trace(go.Scatter3d(
            x=pos[:, 0], y=pos[:, 1], z=pos[:, 2],
            mode='lines',
            line=dict(width=3),
            name=f'Centerline {idx + 1}'
        ))


    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', range=[x_min, x_max]),
            yaxis=dict(title='Y', range=[y_min, y_max]),
            zaxis=dict(title='Z', range=[z_min, z_max]),
            aspectmode='cube',
            # aspectratio=dict(x=1, y=1, z=1)
            # xaxis=dict(title='X'),
            # yaxis=dict(title="Y"),
            # zaxis=dict(title='Z'),
            # aspectmode='data'
        ),
        title=dict(text='CL Visualization', y=0.95, x=0.5, xanchor='center', yanchor='top'),
    )

    fig.show()

In [82]:
poses = np.asarray(poses)
points = poses[0]
x_min, x_max = points[:, 0].min(), points[:, 0].max()
y_min, y_max = points[:, 1].min(), points[:, 1].max()
z_min, z_max = points[:, 2].min(), points[:, 2].max()
print(x_min, x_max)
print(y_min, y_max)
print(z_min, z_max)
plot_cls([poses])

-0.18024094652290087 0.0006284603705091868
-0.0444612188624579 0.007604116491644193
-0.33626436503970797 0.0
0


In [15]:
tracking_freq = 20
progress = tqdm(range(600 * tracking_freq))
n_strands = 5
for i in progress:
    for j in range(5):
        pos, theta = sims[j].step(pos=poses[j], theta=thetas[j])
        poses[j] = pos
        thetas[j] = theta
    if i % tracking_freq == 0:
        progress.set_description(f"Frame {i // tracking_freq}")
        strands_to_one_objs(np.array(poses), i // tracking_freq)

strands_to_one_objs(np.array(poses), 1)
data_to_save = {
    'poses': poses,  # list of numpy arrays
    'labels': strand_labels  # list of dicts
}

np.save("figure_rad_samples.npy", data_to_save, allow_pickle=True)

Frame 599: 100%|██████████| 12000/12000 [36:28<00:00,  5.48it/s]


NameError: name 'go' is not defined

In [ ]:
\text{indices per curl} = 2\pi \cdot \frac{1}{\text{angular\_curl\_freq}}